In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm
import multiprocessing as mp
from transformers import get_linear_schedule_with_warmup

In [ ]:
MAX_LEN = 256//2
BATCH_SIZE = 16*2
EPOCHS =  10//2
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
SEEDS = [42, 123]

In [ ]:
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

In [ ]:
df = pd.read_csv(train_path)
df['rule']= df['rule'].str.lower().str.strip()
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

In [ ]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [ ]:
test_df = pd.read_csv(test_path)
test_df['rule']= test_df.rule.str.lower().str.strip()

augmented_train = add_data(df)
augmented_test = add_data(test_df)

augmented_texts =  df.text.tolist()+augmented_train[0] + augmented_test[0]
augmented_labels =  df.label.tolist()+augmented_train[1] + augmented_test[1]

augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before:{augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',
    'label': 'mean'
})
print('After:',augmented_df.shape)
augmented_df['rule']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

rule_map= {i:j for j,i in enumerate(augmented_df.rule.unique())}
augmented_df['rule_id']= augmented_df.rule.map(rule_map)

augmented_df.head()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

In [ ]:
unlabelled= pd.read_csv('/kaggle/input/jigsaw-unlabelled-14b/sampled_unlabelled_100k_with_predictions.csv')
unlabelled['rule']= unlabelled['rule'].str.lower().str.strip()
unlabelled['text']= unlabelled['rule']+ ' [SEP] '+ unlabelled['body']
unlabelled['rule_id']= unlabelled.rule.map(rule_map)

pseudo_rules = unlabelled.dropna(subset=['rule_id']).rule.unique().tolist()
print(f'Rules in pseudo data: {pseudo_rules}')
print(f'Number of pseudo rules: {len(pseudo_rules)}')

all_rules = augmented_df.rule.unique().tolist()
non_pseudo_rules = [r for r in all_rules if r not in pseudo_rules]
print(f'Rules without pseudo data: {non_pseudo_rules}')

In [ ]:
# unlabelled_taken.sample(frac=.7).groupby(['rule_id','label']).agg('count')

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    for seed in SEEDS:
        train_data, val_data = train_test_split(
            augmented_df, 
            test_size=0.2, 
            stratify=augmented_df["rule"], 
            random_state=seed
        )
        unlabelled['label']= unlabelled.rule_violation.round(0)
        unlabelled= unlabelled.query('text not in @augmented_df.text')
        
        # remove test queries especially
        temp_test_data= test_df["rule"].str.lower().str.strip() + " [SEP] " + test_df["body"]
        unlabelled= unlabelled.query('text not in @temp_test_data')
        print(unlabelled.shape)

        # confident_positives = unlabelled[unlabelled['rule_violation'] >= 0.85]
        # confident_negatives = unlabelled[unlabelled['rule_violation'] <= 0.1]
        
        # n_positives = len(confident_positives)
        # n_negatives_to_sample = int(n_positives * 1.1)
        
        # sampled_negatives = confident_negatives.sample(
        #   n=min(n_negatives_to_sample, len(confident_negatives)),
        #   random_state=seed
        # )
        
        # unlabelled_taken = pd.concat([confident_positives, sampled_negatives], ignore_index=True)
        # unlabelled_taken= unlabelled_taken.sample(frac=.7, random_state=seed)
        # print(f'Seed {seed} - Pseudo: {len(confident_positives)} pos (>=0.85), {len(sampled_negatives)} neg (<=0.1), total={len(unlabelled_taken)}')
        unlabelled_taken= unlabelled.sample(frac=.7,random_state=seed)
        
        train_data.to_csv(f'fixed_train_split_seed_{seed}.csv', index=False)
        val_data.to_csv(f'fixed_val_split_seed_{seed}.csv', index=False)
        unlabelled_taken.to_csv(f'fixed_pseudo_seed_{seed}.csv', index=False)
        print(f'Seed {seed} splits saved: train={len(train_data)}, val={len(val_data)}, pseudo={len(unlabelled_taken)}')
    
    import json
    with open('pseudo_rules.json', 'w') as f:
        json.dump({'pseudo_rules': pseudo_rules, 'non_pseudo_rules': non_pseudo_rules}, f)
    print(f'Saved pseudo_rules.json with {len(pseudo_rules)} pseudo rules and {len(non_pseudo_rules)} non-pseudo rules')

In [ ]:
class JigsawDataset(Dataset):
    def __init__(self, texts, labels,rule_ids, tokenizer, max_len,weights=None):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids
        self.weights = weights if weights is not None else [1.0]*len(texts)

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids']= torch.tensor(self.rule_ids[idx])
        item['weights'] = torch.tensor(self.weights[idx], dtype=torch.float)
        return item

In [ ]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc='Training'):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        logits = model(input_ids, mask)
        weights = batch["weights"].to(device)
        loss = nn.BCEWithLogitsLoss(reduction='none')(logits, labels)
        loss = (loss * weights).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
def validate(model, loader, device):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            rule_ids = batch["rule_ids"]
            
            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)
    
    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)
    
    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}
    
    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]
        
        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            rule_aucs[rule_id] = np.nan
    
    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0
    
    val_loss = total_loss / len(loader)
    return avg_auc_per_rule, val_loss, preds

In [ ]:
def train_model_seed_no_pseudo(seed, gpu_id):
    import random
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    device = torch.device(f"cuda:{gpu_id}")
    print(f"[Seed {seed} NO-PSEUDO] Training on {device}")
    
    train_data = pd.read_csv(f'fixed_train_split_seed_{seed}.csv')
    val_data = pd.read_csv(f'fixed_val_split_seed_{seed}.csv')
    
    train_ds = JigsawDataset(
        train_data['text'].tolist(), 
        train_data['label'].tolist(), 
        train_data['rule_id'].tolist(), 
        tokenizer, MAX_LEN
    )
    
    val_ds = JigsawDataset(
        val_data['text'].tolist(), 
        val_data['label'].tolist(), 
        val_data['rule_id'].tolist(), 
        tokenizer, MAX_LEN
    )
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
    model = JigsawModel(MODEL_PATH).to(device)
    for name, param in model.named_parameters():
        if name.startswith('base.embedding'):
            param.requires_grad = False
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, eps=1e-6)
    total_steps = EPOCHS * len(train_loader)
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    
    best_auc = 0
    best_loss= None
    for epoch in range(EPOCHS):
        print(f"[Seed {seed} NO-PSEUDO] Epoch {epoch+1}/{EPOCHS}")
        loss = train_one_epoch(model, train_loader, optimizer, scheduler, device)
        val_auc, val_loss, val_preds = validate(model, val_loader, device)
        
        print(f"[Seed {seed} NO-PSEUDO] Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
        if val_auc > best_auc:
            best_auc = val_auc
            best_loss= val_loss
            torch.save(model.state_dict(), f"model_no_pseudo_seed_{seed}.bin")
    
    print(f"[Seed {seed} NO-PSEUDO] Best validation AUC: {best_auc:.4f}")
    import json
    with open(f'results_no_pseudo_seed_{seed}.json', 'w') as f:
        json.dump({'seed': seed, 'best_auc': best_auc, 'best_loss': best_loss}, f)

    return seed, best_auc

def train_model_seed_with_pseudo(seed, gpu_id):
    import random
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    device = torch.device(f"cuda:{gpu_id}")
    print(f"[Seed {seed} WITH-PSEUDO] Training on {device}")
    
    train_data = pd.read_csv(f'fixed_train_split_seed_{seed}.csv')
    val_data = pd.read_csv(f'fixed_val_split_seed_{seed}.csv')
    unlabelled_taken = pd.read_csv(f'fixed_pseudo_seed_{seed}.csv')
    
    train_ds = JigsawDataset(
        train_data['text'].tolist()+unlabelled_taken['text'].tolist(), 
        train_data['label'].tolist()+unlabelled_taken['rule_violation'].tolist(), 
        train_data['rule_id'].tolist()+unlabelled_taken['rule_id'].tolist(), 
        tokenizer, MAX_LEN,
        [1.0]*len(train_data) + [.5]*len(unlabelled_taken)
    )
    
    val_ds = JigsawDataset(
        val_data['text'].tolist(), 
        val_data['label'].tolist(), 
        val_data['rule_id'].tolist(), 
        tokenizer, MAX_LEN
    )
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
    model = JigsawModel(MODEL_PATH).to(device)
    for name, param in model.named_parameters():
        if name.startswith('base.embedding'):
            param.requires_grad = False
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, eps=1e-6)
    total_steps = EPOCHS * len(train_loader)
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    
    best_auc = 0
    best_loss= None
    for epoch in range(EPOCHS):
        print(f"[Seed {seed} WITH-PSEUDO] Epoch {epoch+1}/{EPOCHS}")
        loss = train_one_epoch(model, train_loader, optimizer, scheduler, device)
        val_auc, val_loss, val_preds = validate(model, val_loader, device)
        
        print(f"[Seed {seed} WITH-PSEUDO] Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
        if val_auc > best_auc:
            best_auc = val_auc
            best_loss= val_loss
            torch.save(model.state_dict(), f"model_with_pseudo_seed_{seed}.bin")
    
    print(f"[Seed {seed} WITH-PSEUDO] Best validation AUC: {best_auc:.4f}")
    import json
    with open(f'results_with_pseudo_seed_{seed}.json', 'w') as f:
        json.dump({'seed': seed, 'best_auc': best_auc, 'best_loss': best_loss}, f)

    return seed, best_auc

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
      import torch.multiprocessing as mp
      mp.set_start_method('fork', force=True)

      # Train NO-PSEUDO models first
      print("=== Training NO-PSEUDO models ===")
      processes = []
      for idx, seed in enumerate(SEEDS):
         gpu_id = idx % torch.cuda.device_count()
         p = mp.Process(target=train_model_seed_no_pseudo, args=(seed, gpu_id))
         p.start()
         processes.append(p)

      for p in processes:
         p.join()

      # Clear memory
      torch.cuda.empty_cache()
      import gc
      gc.collect()

      # Train WITH-PSEUDO models next
      print("\n=== Training WITH-PSEUDO models ===")
      processes = []
      for idx, seed in enumerate(SEEDS):
         gpu_id = idx % torch.cuda.device_count()
         p = mp.Process(target=train_model_seed_with_pseudo, args=(seed, gpu_id))
         p.start()
         processes.append(p)

      for p in processes:
         p.join()

      import json
      results_no_pseudo = []
      results_with_pseudo = []

      for seed in SEEDS:
        with open(f'results_no_pseudo_seed_{seed}.json', 'r') as f:
            results_no_pseudo.append(json.load(f))
        with open(f'results_with_pseudo_seed_{seed}.json', 'r') as f:
            results_with_pseudo.append(json.load(f))

      aucs_no = [r['best_auc'] for r in results_no_pseudo]
      losses_no = [r['best_loss'] for r in results_no_pseudo]
      aucs_with = [r['best_auc'] for r in results_with_pseudo]
      losses_with = [r['best_loss'] for r in results_with_pseudo]

      print("\n=== NO PSEUDO (for non-pseudo rules) ===")
      print(f"AUC: {np.mean(aucs_no):.4f} ± {np.std(aucs_no):.4f}")
      print(f"Loss: {np.mean(losses_no):.4f} ± {np.std(losses_no):.4f}")

      print("\n=== WITH PSEUDO (for pseudo rules) ===")
      print(f"AUC: {np.mean(aucs_with):.4f} ± {np.std(aucs_with):.4f}")
      print(f"Loss: {np.mean(losses_with):.4f} ± {np.std(losses_with):.4f}")

      print("\nAll 4 models trained!")


In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import json
    with open('pseudo_rules.json', 'r') as f:
        rules_info = json.load(f)
    pseudo_rules = rules_info['pseudo_rules']
    
    df_test = pd.read_csv(test_path)
    df_test['rule'] = df_test['rule'].str.lower().str.strip()
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]
    
    df_test_pseudo = df_test[df_test['rule'].isin(pseudo_rules)].copy()
    df_test_no_pseudo = df_test[~df_test['rule'].isin(pseudo_rules)].copy()
    
    print(f'Test samples with pseudo rules: {len(df_test_pseudo)}')
    print(f'Test samples without pseudo rules: {len(df_test_no_pseudo)}')
    
    device = torch.device("cuda:0")
    
    if len(df_test_no_pseudo) > 0:
        test_ds_no_pseudo = JigsawDataset(
            df_test_no_pseudo['text'].tolist(), 
            [0]*len(df_test_no_pseudo), 
            [0]*len(df_test_no_pseudo), 
            tokenizer, MAX_LEN
        )
        test_loader_no_pseudo = DataLoader(test_ds_no_pseudo, batch_size=BATCH_SIZE)
        
        all_preds_no_pseudo = []
        for seed in SEEDS:
            model = JigsawModel(MODEL_PATH).to(device)
            model.load_state_dict(torch.load(f"model_no_pseudo_seed_{seed}.bin", map_location=device))
            model.eval()
            
            test_preds = []
            with torch.no_grad():
                for batch in tqdm(test_loader_no_pseudo, desc=f"Inference NO-PSEUDO seed {seed}"):
                    ids = batch['input_ids'].to(device)
                    mask = batch['attention_mask'].to(device)
                    logits = model(ids, mask)
                    test_preds.extend(torch.sigmoid(logits).cpu().numpy())
            
            all_preds_no_pseudo.append(test_preds)
        
        ensemble_preds_no_pseudo = np.mean(all_preds_no_pseudo, axis=0)
        df_test_no_pseudo['rule_violation'] = ensemble_preds_no_pseudo
    
    if len(df_test_pseudo) > 0:
        test_ds_pseudo = JigsawDataset(
            df_test_pseudo['text'].tolist(), 
            [0]*len(df_test_pseudo), 
            [0]*len(df_test_pseudo), 
            tokenizer, MAX_LEN
        )
        test_loader_pseudo = DataLoader(test_ds_pseudo, batch_size=BATCH_SIZE)
        
        all_preds_pseudo = []
        for seed in SEEDS:
            model = JigsawModel(MODEL_PATH).to(device)
            model.load_state_dict(torch.load(f"model_with_pseudo_seed_{seed}.bin", map_location=device))
            model.eval()
            
            test_preds = []
            with torch.no_grad():
                for batch in tqdm(test_loader_pseudo, desc=f"Inference WITH-PSEUDO seed {seed}"):
                    ids = batch['input_ids'].to(device)
                    mask = batch['attention_mask'].to(device)
                    logits = model(ids, mask)
                    test_preds.extend(torch.sigmoid(logits).cpu().numpy())
            
            all_preds_pseudo.append(test_preds)
        
        ensemble_preds_pseudo = np.mean(all_preds_pseudo, axis=0)
        df_test_pseudo['rule_violation'] = ensemble_preds_pseudo
    
    df_test_final = pd.concat([df_test_no_pseudo, df_test_pseudo]).sort_index()
    
    sample = pd.read_csv(sample_sub_path)
    sample["rule_violation"] = df_test_final["rule_violation"].values
    sample.to_csv("submission.csv", index=False)
    print(f"Ensembled {len(SEEDS)} models per type (total 4 models)")
else:
    !touch submission.csv
    
!head -n 4 submission.csv

In [ ]:
# 4 models total: 2 seeds × 2 types (with/without pseudo)
# NO-PSEUDO models: trained on original train data only, used for rules without pseudo data
# WITH-PSEUDO models: trained on original train data + pseudo labels, used for 2 rules that have pseudo data
# Inference routes test samples to appropriate model based on their rule

In [ ]:
#ablation
#start                            .8765 3rd epoch .4731 val loss
#all pseudo 
#random sample pseudo 2k          .8871 3nd epoch, .4731 val loss
#random sample pseudo 4k          .8977 .8761 2.5th epoch, .4133 .4663 val loss
#random sample pseudo 8k          .8979 .8932 3.5th epochs, .4223 .4409
#random sample pseudo 16k         .9082 .8904 3th epoch  .4356 .4363 
#random sample pseudo 32k         .8893 .9104 2.5th epoch  .4494 .4286 ***
#random sample pseudo 64k         .8913 3th epoch  .4298
#random sample pseudo 100k        .8913 3th epoch  .4298

# weightitng starts 
#.3 weights

# good options
#. 5 weight, hard labels, remove .1-.85, balance to have same, 20k points taken, dontoverwhelm with pseudo data, for each rule 10k points, so at best have 20k points(10k pairs), at best we have 10k points, so use them with diff wights .5-

In [ ]:
#TODO add softlabelling, add all points without confidence filterineg, weight 
#add weight of .5-.1 for pseudo label
#use mixup 